[Compute LD-scores for annotations](https://github.com/omerwe/polyfun/wiki/2.-Using-and-creating-functional-annotations#computing-ld-scores-for-annotations)

Download baseline-LF2.2 annotations for 19 million SNPs

In [ ]:
%%bash

# download and decompress
curl -O https://broad-alkesgroup-ukbb-ld.s3.amazonaws.com/UKBB_LD/baselineLF_v2.2.UKB.polyfun.tar.gz
tar xzf baselineLF_v2.2.UKB.polyfun.tar.gz
rm baselineLF_v2.2.UKB.polyfun.tar.gz



Download ACAF PLINK bfiles.

In [ ]:
%%bash

gcloud storage cp --billing-project=$GOOGLE_PROJECT gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr11.* .

Get list of SNPs from annotation file

In [ ]:
import pandas as pandas

df = pd.read_parquet("baselineLF2.2.UKB/baselineLF2.2.UKB.11.annot.parquet")
df["SNP"].dropna().to_csv("baselineLF2.2.UKB.11.snplist", index=False, header=False)

dbSNP references files to map `CPID -> rsID`

In [ ]:
%%bash
# GRCh38/hg38/b38
wget http://fileserve.mrcieu.ac.uk/dbsnp/dbsnp.v153.hg38.vcf.gz .
wget http://fileserve.mrcieu.ac.uk/dbsnp/dbsnp.v153.hg38.vcf.gz.tbi .

gcloud storage cp --billing-project=$GOOGLE_PROJECT dbsnp.v153.hg38.vcf.gz $WORKSPACE_BUCKET/data/reference

In [ ]:
cat chr22.bim | awk 'OFS = "\t" {print $1, $4}' > chr22.regions

bcftools query \
--regions-file chr22.regions \
--print-header \
--format '%CHROM\t%POS\t%ID\t%REF\t%ALT{0}\n' \
dbsnp.v153.hg38.vcf.gz > dbsnp-chr22-chr_pos_rsid.tsv

Annotated regions to extract from genotypes

In [ ]:
dbsnp = pd.read_csv("dbsnp-chr22-chr_pos_rsid.tsv", sep='\t')

# find dbSNP rows in the annotation files
dbsnp_annotated = dbsnp[dbsnp['[3]ID'].isin(df['SNP'])]

annotated_regions = dbsnp_annotated.loc[:, ['# [1]CHROM', '[2]POS', '[2]POS']]
annotated_regions = dbsnp_annotated.rename(columns={
  '# [1]CHROM': 'CHROM',
  '[2]POS': 'START'
}).assign(END=lambda df: df['START'])[["CHROM", "START", "END"]]

annotated_regions.to_csv('chr22_annotated.ranges', sep='\t', index=False)



In [ ]:
bim = pd.read_csv('chr22.bim', sep='\t', header=None, names=['CHR', 'ID', 'CM', 'POS', 'A1', 'A2'])

mapping files to convert from genotype files build to annotation files build

In [ ]:
# update IDs
update_names1 = pd.merge(
    bim,
    dbsnp,
    left_on=['CHR', 'POS', 'A1', 'A2'],
    right_on=['# [1]CHROM', '[2]POS', '[4]REF', '[5]ALT'],
    how='inner'
)[['ID', '[3]ID']]

update_names2 = pd.merge(
    bim,
    dbsnp,
    left_on=['CHR', 'POS', 'A2', 'A1'],
    right_on=['# [1]CHROM', '[2]POS', '[4]REF', '[5]ALT'],
    how='inner'
)[['ID', '[3]ID']]

update_names = pd.concat([update_names1, update_names2], axis=0, ignore_index=True)
update_names_dedup = update_names[~update_names.duplicated('ID')]

update_names_dedup.to_csv("update_names.txt", header = False, index = False, sep = '\t')


In [ ]:
%%bash

plink2 \
--bfile chr11 \
--extract baselineLF2.2.UKB.11.snplist \
--make-bed \
--bfile chr22 \
--keep ancestry_ref_panel_v8.id \
--extract range chr22_annotated.ranges \
--update-name update_names.txt \
--out chr11-rsids-annotated